# Geração de Amostras V6 — Pipeline em 2 Fases (Jogo dos Pontinhos)

Notebook **autônomo** que gera o dataset supervisionado para o tabuleiro 9×7
(4×3 caixas) em duas fases:

- **Fase 1 — Geração de estados.** 95% das amostras vêm de autoplay
  Minimax(p=3) × Minimax(p=3); 5% são tabuleiros aleatórios. A distribuição
  de traços segue a estratificação acordada com o usuário.
- **Fase 2 — Cálculo da melhor jogada.** Para cada estado distinto, roda
  Minimax(p=7) e grava `rotulos` + `scores` no respectivo NPZ. Duplicatas
  reusam o resultado do estado idêntico (cache global por hash).

**Saída:** `dados/profundidade_minmax_7_corrigido/dataset_pequeno_NNNN.npz`,
estrutura idêntica ao NPZ de referência (`dataset_pequeno_0001.npz`):

| Campo | Shape | Dtype |
|---|---|---|
| `estados` | `(N, 9, 7)` | `int8` (domínio `{0,1,8,9}`) |
| `rotulos` | `(N,)` | `<U5` |
| `scores` | `(N, 31)` | `float32` |
| `generation_mode` | `(N,)` | `int8` (`0`=aleatório, `3`=Minimax p=3 vs p=3) |
| `labels_canonicos` | `(31,)` | `<U5` |
| `minimax_depth` | `(1,)` | `int32` |

**Retomada:** o estado é reconstruído lendo o diretório de NPZ — não há
sidecar JSON. Se o processo for interrompido, basta re-executar.

**Multiprocessing:** workers ficam em
`gerador_dados/jogo_pontinhos/gerador_amostras_v6_pontinhos.py` (necessário
para o `spawn` do Windows).


In [1]:
# %load_ext autoreload
# %autoreload 2

import os
import sys
import time
import math
import json
import hashlib
import concurrent.futures
import multiprocessing as mp
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# Adiciona a raiz do repositorio ao sys.path para importar o pacote backend
ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "gerador_dados").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Nao encontrei a raiz do repositorio (pasta 'gerador_dados').")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from gerador_dados.jogo_pontinhos.gerador_amostras_v6_pontinhos import (
    gerar_amostra,
    calcular_scores,
    LABELS_CANONICOS,
    N_LABELS,
    ALTURA,
    LARGURA,
    SCORE_INDISPONIVEL,
    MODO_ALEATORIO,
    MODO_AUTOPLAY,
)

print(f"ROOT = {ROOT}")
print(f"N_LABELS = {N_LABELS}")
print(f"ALTURA x LARGURA = {ALTURA} x {LARGURA}")


ROOT = D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend
N_LABELS = 31
ALTURA x LARGURA = 9 x 7


## 1. Configuração

Quotas (501.500 estados **distintos** no total) e parâmetros de pool. Edite
`OUTPUT_DIR` se quiser direcionar para outra pasta.

| Faixa de traços | Quota (distintos) | % do total |
|---|---:|---:|
| 5–11 (abertura) | 55.000 | 10,97% |
| 12–17 (1ª metade) | 160.000 | 31,90% |
| 18–23 (2ª metade) | 220.000 | 43,87% |
| 24–28 (fase quente) | 66.000 | 13,16% |
| 29–30 (final) | 500 | 0,10% |
| **Total** | **501.500** | **100,00%** |


In [ ]:
# Faixas como tuplas (faixa_min, faixa_max, alvo_distintos)
FAIXAS = [
    (5,  11,  55_000),
    (12, 17, 160_000),
    (18, 23, 220_000),
    (24, 28,  66_000),
    (29, 30,     496),
]

PROFUNDIDADE_AUTOPLAY = 2   # Fase 1: Minimax(p=2) x Minimax(p=2)
PROFUNDIDADE_SCORING  = 7   # Fase 2: Minimax(p=7) por estado

FRACAO_ALEATORIO = 0.05     # 5% modo aleatorio, 95% autoplay

TAMANHO_LOTE = 5_000        # estados por NPZ
SEED_GLOBAL  = 20260508     # determinismo dos sorteios no main; workers tem seed proprio

OUTPUT_DIR = ROOT / "dados" / f"profundidade_minmax_{PROFUNDIDADE_SCORING}_corrigido"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_WORKERS = max(1, mp.cpu_count() - 2)   # Ryzen 7 5700X => 14 workers, 2 cores livres
QUEUE_DEPTH = NUM_WORKERS * 4              # quantos futuros manter no ar

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"NUM_WORKERS = {NUM_WORKERS}")
print(f"Total alvo de distintos = {sum(f[2] for f in FAIXAS):,}")


OUTPUT_DIR = D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_7_corrigido
NUM_WORKERS = 14
Total alvo de distintos = 501,500


## 2. Recuperação do estado a partir do diretório

Lê todos os NPZs já existentes para reconstruir:
- O **set global de hashes** (chave: `estados[i].tobytes()`) — fonte da
  verdade para distinguir duplicatas.
- A **contagem de distintos por faixa** — para saber onde retomar.
- O **índice do último NPZ** — para continuar a numeração.

A faixa de cada estado é derivada do número de traços (`np.sum(estado == 9)`).


In [4]:
def faixa_de_estado(num_tracos: int) -> int | None:
    """Retorna o indice da faixa em FAIXAS, ou None se fora de qualquer faixa."""
    for i, (lo, hi, _alvo) in enumerate(FAIXAS):
        if lo <= num_tracos <= hi:
            return i
    return None


def proximo_indice_npz(output_dir: Path) -> int:
    npzs = sorted(output_dir.glob("dataset_pequeno_*.npz"))
    if not npzs:
        return 0
    ultimo = npzs[-1].stem.split("_")[-1]
    try:
        return int(ultimo)
    except ValueError:
        return 0


def reconstruir_estado_global(output_dir: Path):
    """Le todos os NPZs e devolve (hashes_set, contagens_por_faixa, ultimo_idx)."""
    hashes: set[bytes] = set()
    contagens = [0] * len(FAIXAS)
    npzs = sorted(output_dir.glob("dataset_pequeno_*.npz"))
    for path in npzs:
        try:
            d = np.load(path)
        except Exception as e:
            print(f"[WARN] NPZ corrompido ignorado: {path.name} ({e})")
            continue
        estados = d["estados"]
        for i in range(estados.shape[0]):
            chave = estados[i].tobytes()
            num_tracos = int(np.sum(estados[i] == 9))
            faixa_idx = faixa_de_estado(num_tracos)
            if chave not in hashes:
                hashes.add(chave)
                if faixa_idx is not None:
                    contagens[faixa_idx] += 1
    ultimo_idx = proximo_indice_npz(output_dir)
    return hashes, contagens, ultimo_idx


HASHES_GLOBAIS, CONTAGEM_DISTINTOS, ULTIMO_NPZ_IDX = reconstruir_estado_global(OUTPUT_DIR)

print(f"NPZs existentes: {len(sorted(OUTPUT_DIR.glob('dataset_pequeno_*.npz')))}")
print(f"Estados distintos ja gerados: {len(HASHES_GLOBAIS):,}")
print(f"Ultimo indice NPZ: {ULTIMO_NPZ_IDX}")
print()
print("Progresso por faixa:")
for i, (lo, hi, alvo) in enumerate(FAIXAS):
    feito = CONTAGEM_DISTINTOS[i]
    pct = (feito / alvo * 100) if alvo else 0
    marca = "OK" if feito >= alvo else " ."
    print(f"  [{marca}] faixa {lo}-{hi}: {feito:>7,} / {alvo:>7,}  ({pct:5.1f}%)")


NPZs existentes: 1
Estados distintos ja gerados: 4,998
Ultimo indice NPZ: 0

Progresso por faixa:
  [ .] faixa 5-11:   4,998 /  55,000  (  9.1%)
  [ .] faixa 12-17:       0 / 160,000  (  0.0%)
  [ .] faixa 18-23:       0 / 220,000  (  0.0%)
  [ .] faixa 24-28:       0 /  66,000  (  0.0%)
  [ .] faixa 29-30:       0 /     500  (  0.0%)


## 3. Fase 1 — Geração de estados

Para cada faixa em ordem:
1. Mantém uma fila de `QUEUE_DEPTH` futuros submetidos ao pool.
2. Para cada futuro completo: incrementa o contador de distintos se novo,
   acumula no buffer (incluindo duplicatas).
3. A cada `TAMANHO_LOTE = 5.000` amostras no buffer, grava NPZ atomicamente
   (`*.tmp` → `os.replace`).
4. Quando a faixa atinge a quota de distintos, drena os futuros pendentes e
   passa para a próxima faixa.

NPZs da Fase 1 já têm a estrutura completa, mas com `rotulos = ""` e
`scores = -1e9` em todas as posições. A Fase 2 sobrescreve com os valores
calculados pelo Minimax(p=7).


In [5]:
def _converter_buffer_para_arrays(buffer_estados, buffer_modos):
    n = len(buffer_estados)
    estados_arr = np.empty((n, ALTURA, LARGURA), dtype=np.int8)
    for i, b in enumerate(buffer_estados):
        estados_arr[i] = np.frombuffer(b, dtype=np.int8).reshape(ALTURA, LARGURA)
    modos_arr = np.array(buffer_modos, dtype=np.int8)
    return estados_arr, modos_arr


def salvar_npz_fase1(buffer_estados, buffer_modos, idx_npz: int):
    """Grava um NPZ contendo apenas estados + generation_mode preenchidos.

    rotulos/scores ficam como placeholder (vazio / -1e9) — a Fase 2 reescreve.
    Sobrescrita atomica via .tmp + os.replace.
    """
    estados_arr, modos_arr = _converter_buffer_para_arrays(buffer_estados, buffer_modos)
    n = estados_arr.shape[0]
    rotulos_arr = np.empty(n, dtype="<U5")
    rotulos_arr[:] = ""
    scores_arr = np.full((n, N_LABELS), SCORE_INDISPONIVEL, dtype=np.float32)
    labels_arr = np.array(LABELS_CANONICOS, dtype="<U5")
    depth_arr  = np.array([PROFUNDIDADE_SCORING], dtype=np.int32)

    nome = f"dataset_pequeno_{idx_npz:04d}.npz"
    caminho = OUTPUT_DIR / nome
    tmp = caminho.with_name(caminho.stem + ".tmp.npz")  # mantem sufixo .npz para nao virar .npz.tmp.npz
    np.savez_compressed(
        tmp,
        estados=estados_arr,
        rotulos=rotulos_arr,
        scores=scores_arr,
        generation_mode=modos_arr,
        labels_canonicos=labels_arr,
        minimax_depth=depth_arr,
    )
    os.replace(tmp, caminho)
    return caminho


def fase1_gerar(hashes_globais, contagem_distintos, ultimo_idx):
    """Loop principal da Fase 1. Atualiza os tres argumentos in-place."""
    rng_main = np.random.default_rng(SEED_GLOBAL)

    buffer_estados: list[bytes] = []
    buffer_modos:   list[int]   = []
    npz_idx_ref = [ultimo_idx]

    def _flush_se_cheio():
        if len(buffer_estados) >= TAMANHO_LOTE:
            npz_idx_ref[0] += 1
            cam = salvar_npz_fase1(buffer_estados[:TAMANHO_LOTE],
                                   buffer_modos[:TAMANHO_LOTE],
                                   npz_idx_ref[0])
            del buffer_estados[:TAMANHO_LOTE]
            del buffer_modos[:TAMANHO_LOTE]
            return cam
        return None

    inicio_geral = time.time()
    with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for faixa_idx, (lo, hi, alvo) in enumerate(FAIXAS):
            if contagem_distintos[faixa_idx] >= alvo:
                print(f"[FAIXA {lo}-{hi}] ja completa ({contagem_distintos[faixa_idx]:,}/{alvo:,}). Pulando.")
                continue

            print(f"\n[FAIXA {lo}-{hi}] alvo={alvo:,} distintos. Iniciando.")
            inicio_faixa = time.time()
            distintos_adicionados_inicio = contagem_distintos[faixa_idx]
            futuros: dict = {}

            def _submeter():
                modo = MODO_ALEATORIO if rng_main.random() < FRACAO_ALEATORIO else MODO_AUTOPLAY
                seed_w = int(rng_main.integers(0, 2**31 - 1))
                args = (lo, hi, modo, PROFUNDIDADE_AUTOPLAY, seed_w)
                f = executor.submit(gerar_amostra, args)
                futuros[f] = args

            # Burst inicial
            for _ in range(QUEUE_DEPTH):
                _submeter()

            recebidos_faixa = 0
            ultimo_log = time.time()
            while contagem_distintos[faixa_idx] < alvo:
                done, _pending = concurrent.futures.wait(
                    list(futuros.keys()),
                    return_when=concurrent.futures.FIRST_COMPLETED,
                )
                for f in done:
                    futuros.pop(f)
                    estado_bytes, modo, _K = f.result()
                    buffer_estados.append(estado_bytes)
                    buffer_modos.append(modo)
                    recebidos_faixa += 1
                    if estado_bytes not in hashes_globais:
                        hashes_globais.add(estado_bytes)
                        contagem_distintos[faixa_idx] += 1
                    cam = _flush_se_cheio()
                    if cam is not None:
                        print(f"  [{lo}-{hi}] NPZ salvo: {cam.name} | "
                              f"distintos={contagem_distintos[faixa_idx]:,}/{alvo:,} | "
                              f"recebidos_faixa={recebidos_faixa:,}")
                    if contagem_distintos[faixa_idx] >= alvo:
                        break
                    _submeter()

                # Log periodico (cada ~30 s)
                if time.time() - ultimo_log > 30:
                    elapsed = time.time() - inicio_faixa
                    novos_dist = contagem_distintos[faixa_idx] - distintos_adicionados_inicio
                    taxa = novos_dist / elapsed if elapsed > 0 else 0
                    restante = max(0, alvo - contagem_distintos[faixa_idx])
                    eta = (restante / taxa) if taxa > 0 else float("inf")
                    print(f"  [{lo}-{hi}] +{novos_dist:,} distintos em {elapsed:6.1f}s "
                          f"({taxa:6.1f}/s) | ETA faixa: {eta:7.1f}s")
                    ultimo_log = time.time()

            # Drena futuros pendentes (ainda contam para o NPZ atual)
            for f in concurrent.futures.as_completed(list(futuros.keys())):
                estado_bytes, modo, _K = f.result()
                buffer_estados.append(estado_bytes)
                buffer_modos.append(modo)
                if estado_bytes not in hashes_globais:
                    hashes_globais.add(estado_bytes)
                    fi = faixa_de_estado(int(np.sum(
                        np.frombuffer(estado_bytes, dtype=np.int8).reshape(ALTURA, LARGURA) == 9
                    )))
                    if fi is not None:
                        contagem_distintos[fi] += 1
                cam = _flush_se_cheio()
                if cam is not None:
                    print(f"  [{lo}-{hi}] NPZ (drain) salvo: {cam.name}")
            futuros.clear()

            elapsed = time.time() - inicio_faixa
            print(f"[FAIXA {lo}-{hi}] DONE em {elapsed:.1f}s. "
                  f"distintos={contagem_distintos[faixa_idx]:,}/{alvo:,}")

    # Flush final do que sobrou no buffer
    if buffer_estados:
        npz_idx_ref[0] += 1
        cam = salvar_npz_fase1(buffer_estados, buffer_modos, npz_idx_ref[0])
        print(f"\n[FINAL] NPZ residual salvo: {cam.name} ({len(buffer_estados)} amostras)")
        buffer_estados.clear()
        buffer_modos.clear()

    print(f"\nFase 1 concluida em {time.time() - inicio_geral:.1f}s. "
          f"NPZs ate agora: {npz_idx_ref[0]}")
    return npz_idx_ref[0]


# Para o multiprocessing funcionar no Jupyter/Windows e necessario que o pool
# seja criado a partir do __main__ (ja somos). Basta executar:
ULTIMO_NPZ_IDX = fase1_gerar(HASHES_GLOBAIS, CONTAGEM_DISTINTOS, ULTIMO_NPZ_IDX)



[FAIXA 5-11] alvo=55,000 distintos. Iniciando.
  [5-11] +0 distintos em   30.1s (   0.0/s) | ETA faixa:     infs
  [5-11] NPZ salvo: dataset_pequeno_0001.npz | distintos=4,999/55,000 | recebidos_faixa=5,000
  [5-11] +3,940 distintos em   60.1s (  65.6/s) | ETA faixa:   702.6s
  [5-11] NPZ salvo: dataset_pequeno_0002.npz | distintos=9,990/55,000 | recebidos_faixa=10,000
  [5-11] +8,426 distintos em   90.1s (  93.5/s) | ETA faixa:   444.6s
  [5-11] NPZ salvo: dataset_pequeno_0003.npz | distintos=14,977/55,000 | recebidos_faixa=15,000
  [5-11] +12,758 distintos em  120.1s ( 106.2/s) | ETA faixa:   350.6s
  [5-11] NPZ salvo: dataset_pequeno_0004.npz | distintos=19,967/55,000 | recebidos_faixa=20,000
  [5-11] +17,138 distintos em  150.1s ( 114.2/s) | ETA faixa:   287.9s
  [5-11] NPZ salvo: dataset_pequeno_0005.npz | distintos=24,939/55,000 | recebidos_faixa=25,000
  [5-11] +21,569 distintos em  180.1s ( 119.7/s) | ETA faixa:   237.5s
  [5-11] NPZ salvo: dataset_pequeno_0006.npz | distintos

KeyboardInterrupt: 

## 4. Fase 2 — Cálculo da melhor jogada (Minimax p=7)

1. Varre todos os NPZs em `OUTPUT_DIR` e separa os que ainda não têm
   `rotulos` preenchidos.
2. Coleta o **set único** de estados (chave = `bytes`) entre todos esses
   NPZs — duplicatas são processadas apenas uma vez.
3. Em paralelo: para cada estado único, calcula `(rotulo, scores[31])` via
   `melhor_jogada_com_scores(estado, profundidade=7)`. Slots inválidos
   recebem `-1e9`. Empates no argmax são resolvidos com escolha aleatória
   (já feito por `melhor_jogada_com_scores`).
4. Reescreve cada NPZ atomicamente: mantém `estados` e `generation_mode`,
   substitui `rotulos` e `scores` pelos valores calculados.


In [ ]:
def _npz_ja_processado(d: np.lib.npyio.NpzFile) -> bool:
    """Heuristica: se o primeiro rotulo for nao-vazio, o NPZ ja foi processado."""
    rot = d["rotulos"]
    return rot.size > 0 and rot[0] != ""


def coletar_estados_pendentes(output_dir: Path):
    """Retorna (estados_unicos: set[bytes], npzs_pendentes: list[Path])."""
    estados_unicos: set[bytes] = set()
    pendentes: list[Path] = []
    for path in sorted(output_dir.glob("dataset_pequeno_*.npz")):
        with np.load(path) as d:
            if _npz_ja_processado(d):
                continue
            pendentes.append(path)
            estados = d["estados"]
            for i in range(estados.shape[0]):
                estados_unicos.add(estados[i].tobytes())
    return estados_unicos, pendentes


def calcular_cache_scores(estados_unicos: set[bytes], profundidade: int):
    """Roda Minimax(profundidade) em paralelo para cada estado unico."""
    cache: dict[bytes, tuple[str, np.ndarray]] = {}
    items = list(estados_unicos)
    n_total = len(items)
    if n_total == 0:
        return cache
    inicio = time.time()
    ultimo_log = inicio
    with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futuros = {
            executor.submit(calcular_scores, (eb, profundidade)): eb
            for eb in items
        }
        n_done = 0
        for f in concurrent.futures.as_completed(futuros):
            eb = futuros[f]
            cache[eb] = f.result()
            n_done += 1
            agora = time.time()
            if agora - ultimo_log > 15 or n_done == n_total:
                taxa = n_done / (agora - inicio) if agora > inicio else 0
                eta = (n_total - n_done) / taxa if taxa > 0 else float("inf")
                print(f"  scoring: {n_done:>7,}/{n_total:>7,} "
                      f"({taxa:6.2f}/s) | ETA: {eta/60:5.1f} min")
                ultimo_log = agora
    return cache


def reescrever_npz_com_scores(path: Path, cache: dict):
    with np.load(path) as d:
        estados = d["estados"]
        modos = d["generation_mode"]
        labels_can = d["labels_canonicos"]
        depth = d["minimax_depth"]
    n = estados.shape[0]
    rotulos_novos = np.empty(n, dtype="<U5")
    scores_novos = np.full((n, N_LABELS), SCORE_INDISPONIVEL, dtype=np.float32)
    for i in range(n):
        rotulo, scores = cache[estados[i].tobytes()]
        rotulos_novos[i] = rotulo
        scores_novos[i] = scores
    tmp = path.with_name(path.stem + ".tmp.npz")  # mantem sufixo .npz
    np.savez_compressed(
        tmp,
        estados=estados,
        rotulos=rotulos_novos,
        scores=scores_novos,
        generation_mode=modos,
        labels_canonicos=labels_can,
        minimax_depth=depth,
    )
    os.replace(tmp, path)


def fase2_scoring():
    estados_unicos, pendentes = coletar_estados_pendentes(OUTPUT_DIR)
    print(f"NPZs pendentes de scoring: {len(pendentes)}")
    print(f"Estados unicos a processar: {len(estados_unicos):,}")
    if not pendentes:
        print("Nada a fazer.")
        return
    print(f"Calculando Minimax(p={PROFUNDIDADE_SCORING}) em {NUM_WORKERS} workers...")
    cache = calcular_cache_scores(estados_unicos, PROFUNDIDADE_SCORING)
    print(f"\nReescrevendo {len(pendentes)} NPZs...")
    for i, path in enumerate(pendentes, 1):
        reescrever_npz_com_scores(path, cache)
        print(f"  [{i:>3}/{len(pendentes)}] {path.name}")
    print(f"\nFase 2 concluida.")


fase2_scoring()


## 5. Auditoria final

Sanidade pós-pipeline:
- Domínio do tensor: cada `estado` deve ficar em `{0, 1, 8, 9}`.
- `rotulos` não-vazios em todos os NPZs.
- `scores` por amostra: máximo finito (não pode ser `-1e9` em todas as 31
  posições, exceto se o estado for terminal).
- Distribuição empírica de `generation_mode`.
- Distribuição empírica por faixa de traços.
- Total de estados distintos ≥ 501.500.


In [ ]:
def auditoria(output_dir: Path):
    npzs = sorted(output_dir.glob("dataset_pequeno_*.npz"))
    print(f"NPZs encontrados: {len(npzs)}")
    if not npzs:
        return

    total = 0
    distintos: set[bytes] = set()
    contador_modo = Counter()
    contador_faixa = Counter()
    rotulos_vazios = 0
    estados_terminais = 0
    valores_estado_globais: set[int] = set()
    scores_min = math.inf
    scores_max = -math.inf

    for path in npzs:
        with np.load(path) as d:
            estados = d["estados"]
            rotulos = d["rotulos"]
            scores = d["scores"]
            modos = d["generation_mode"]
            n = estados.shape[0]
            total += n
            valores_estado_globais.update(np.unique(estados).tolist())
            for i in range(n):
                chave = estados[i].tobytes()
                distintos.add(chave)
                num_t = int(np.sum(estados[i] == 9))
                fi = faixa_de_estado(num_t)
                if fi is not None:
                    contador_faixa[(FAIXAS[fi][0], FAIXAS[fi][1])] += 1
                else:
                    contador_faixa[("fora_de_faixa", num_t)] += 1
                contador_modo[int(modos[i])] += 1
                if rotulos[i] == "":
                    rotulos_vazios += 1
                # Score sanity (apenas valores finitos != -1e9)
                validos = scores[i][scores[i] > SCORE_INDISPONIVEL / 2]
                if validos.size == 0:
                    estados_terminais += 1
                else:
                    scores_min = min(scores_min, float(validos.min()))
                    scores_max = max(scores_max, float(validos.max()))

    print(f"\nTotal de amostras (incluindo duplicatas): {total:,}")
    print(f"Total de estados distintos: {len(distintos):,}")
    print(f"Valores unicos em 'estados' (todos os NPZs): {sorted(valores_estado_globais)}")
    print(f"  (esperado: [0, 1, 8, 9])")
    print(f"Rotulos vazios: {rotulos_vazios:,}  (esperado: 0 ou == nro de estados terminais)")
    print(f"Estados terminais (nenhuma jogada valida): {estados_terminais}")
    print(f"Scores validos: min={scores_min}, max={scores_max}")

    print(f"\nDistribuicao por generation_mode:")
    for k, v in sorted(contador_modo.items()):
        nome = {0: 'aleatorio', 3: 'autoplay_p=3'}.get(k, f'modo_{k}')
        print(f"  {k} ({nome}): {v:>9,}  ({v/total*100:5.2f}%)")

    print(f"\nDistribuicao por faixa de tracos:")
    for (lo, hi, alvo) in FAIXAS:
        v = contador_faixa.get((lo, hi), 0)
        print(f"  {lo}-{hi}: {v:>9,}  alvo distintos={alvo:,}")
    fora = {k: v for k, v in contador_faixa.items() if isinstance(k[0], str)}
    if fora:
        print(f"  [fora de faixa]: {sum(fora.values())} amostras  detalhe={dict(fora)}")


auditoria(OUTPUT_DIR)
